In [25]:
!git clone https://github.com/hochu-shunsuke/gobblet-gobblers.git
%cd gobblet-gobblers

Cloning into 'gobblet-gobblers'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 44 (delta 8), reused 43 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 115.34 KiB | 3.60 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/gobblet-gobblers/src/gobblet-gobblers/src/gobblet-gobblers


In [26]:
import json
with open('data/training_data_20251230_213624.json') as f:
    games = json.load(f)
print(f"📊 {len(games)} games loaded!")

📊 1500 games loaded!


In [27]:
# ============================================
# 訓練コード（高品質データ版）
# ============================================
!pip install torch numpy -q

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import Dataset, DataLoader

print("✅ PyTorch ready!")

# データ前処理
def board_to_tensor(state):
    tensor = np.zeros((6, 3, 3), dtype=np.float32)
    for row in range(3):
        for col in range(3):
            stack = state['board'][row][col]
            if stack:
                piece_str = stack[-1]
                player = 0 if piece_str[0] == '1' else 1
                size = {'S': 0, 'M': 1, 'L': 2}[piece_str[1]]
                channel = player * 3 + size
                tensor[channel, row, col] = 1.0
    return tensor

def move_to_index(move):
    size_map = {'SMALL': 0, 'MEDIUM': 1, 'LARGE': 2}
    size_idx = size_map.get(move['size'], 0)
    to_idx = move['to'][0] * 3 + move['to'][1]
    if move['from'] is None:
        return size_idx * 9 + to_idx
    else:
        return 27 + size_idx * 18 + to_idx

# 学習データ作成
X = []
y = []
for game in games:
    if game['winner'] is None:
        continue
    winner = game['winner']
    for i, move in enumerate(game['moves']):
        if move['player'] == winner and i < len(game['states']) - 1:
            try:
                state = game['states'][i]
                X.append(board_to_tensor(state))
                y.append(min(move_to_index(move), 80))
            except:
                pass

X = np.array(X)
y = np.array(y)
print(f"✅ Training samples: {len(X)}")

✅ PyTorch ready!
✅ Training samples: 8452


In [28]:
# ネットワーク定義（V2: より深い）
class GobbletNetV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(6, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv3 = nn.Conv2d(128, 128, 3, padding=1)
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 81)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.relu(self.conv3(x))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# DataLoader
class GameDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = GameDataset(X, y)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_set, val_set = torch.utils.data.random_split(dataset, [train_size, val_size])
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=64)

# 訓練
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GobbletNetV2().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"🖥️ Device: {device}")
print(f"📊 Train: {train_size}, Val: {val_size}")
print("\n🚀 Training...")

EPOCHS = 50
for epoch in range(EPOCHS):
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        model.eval()
        correct = sum((model(xb.to(device)).argmax(1) == yb.to(device)).sum().item()
                      for xb, yb in val_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} | Val Acc: {100*correct/val_size:.1f}%")

print("\n✅ Training complete!")

🖥️ Device: cpu
📊 Train: 6761, Val: 1691

🚀 Training...
Epoch 10/50 | Val Acc: 86.8%
Epoch 20/50 | Val Acc: 88.4%
Epoch 30/50 | Val Acc: 88.6%
Epoch 40/50 | Val Acc: 88.6%
Epoch 50/50 | Val Acc: 88.9%

✅ Training complete!


In [29]:
# モデル保存
torch.save(model.state_dict(), 'gobblet_model_v2.pt')
print("💾 Saved: gobblet_model_v2.pt")

from google.colab import files
files.download('gobblet_model_v2.pt')

💾 Saved: gobblet_model_v2.pt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>